# The Simpsons viewer Analytics - Second practical work

## Data Visualization

**Author:** Alexis Vendrix

**Date:** 20/04/2026



This document contains the data processing pipeline and the analytical design process for *The Simpsons* viewers dataset.



### Libraries

In [2]:
import pandas as pd
import numpy as np
import altair as alt

## 1. Data Processing and Cleaning

In [3]:
df_char = pd.read_csv("data/simpsons_characters.csv")
df_script = pd.read_csv("data/simpsons_script_lines.csv")

/var/folders/v5/jx4rcwvs44bd_b45d0yvr_8w0000gn/T/ipykernel_80919/3293692334.py:2: DtypeWarning: Columns (4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_script = pd.read_csv("data/simpsons_script_lines.csv")


In [ ]:
df_script.head(20)
df_script = df_script[df_script['character_id'].notnull()]
df_script.head(20)


,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,True,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
5,9554,32,214,Martin Prince: (HOARSE WHISPER) I don't think ...,877000,True,38.0,3.0,Martin Prince,Springfield Elementary School,I don't think there's anything left to say.,i dont think theres anything left to say,8
6,9555,32,215,Edna Krabappel-Flanders: Bart?,881000,True,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,Bart?,bart,1
7,9556,32,216,Bart Simpson: Victory party under the slide!,882000,True,8.0,3.0,Bart Simpson,Springfield Elementary School,Victory party under the slide!,victory party under the slide,5
9,9558,32,218,Lisa Simpson: (CALLING) Mr. Bergstrom! Mr. Ber...,889000,True,9.0,374.0,Lisa Simpson,Apartment Building,Mr. Bergstrom! Mr. Bergstrom!,mr bergstrom mr bergstrom,4
10,9559,32,219,"Landlady: Hey, hey, he Moved out this morning....",893000,True,469.0,374.0,Landlady,Apartment Building,"Hey, hey, he Moved out this morning. He must h...",hey hey he moved out this morning he must have...,19


<class 'pandas.core.frame.DataFrame'>
Index: 140750 entries, 0 to 158270
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  140750 non-null  int64  
 1   episode_id          140750 non-null  int64  
 2   number              140750 non-null  int64  
 3   raw_text            140750 non-null  object 
 4   timestamp_in_ms     140750 non-null  object 
 5   speaking_line       140750 non-null  object 
 6   character_id        140750 non-null  object 
 7   location_id         140343 non-null  float64
 8   raw_character_text  140749 non-null  object 
 9   raw_location_text   140342 non-null  object 
 10  spoken_words        132110 non-null  object 
 11  normalized_text     132085 non-null  object 
 12  word_count          132110 non-null  object 
dtypes: float64(1), int64(3), object(9)
memory usage: 15.0+ MB


In [30]:


# 1. Clean Data Types
# Convert word_count to float/int so we can sum it
df_script['word_count'] = pd.to_numeric(df_script['word_count'], errors='coerce')

# Convert character_id to numeric to ensure it matches the ID in df_characters
df_script['character_id'] = pd.to_numeric(df_script['character_id'], errors='coerce')

# 2. Aggregate: Sum word_count by character_id
char_words = df_script.groupby('character_id')['word_count'].sum().reset_index()

# 3. Join: Bring in the character names
# We use a left join on char_words to ensure we keep our calculated sums
top_20 = pd.merge(
    char_words, 
    df_char[['id', 'name']], 
    left_on='character_id', 
    right_on='id'
)

# 4. Final Polish: Sort and Slice
top_20 = top_20.sort_values(by='word_count', ascending=False).head(20)

# Optional: Clean up the extra ID column and reset index for a clean table
top_20 = top_20.drop(columns=['id']).reset_index(drop=True)

top_20

,character_id,word_count,name
0,1.0,1270666.0,Marge Simpson
1,241.0,1154000.0,Entire Town
2,2.0,682780.0,Homer Simpson
3,3592.0,672126.0,Robert Pinsky
4,2739.0,571000.0,ABBA
5,9.0,364576.0,Lisa Simpson
6,8.0,218814.0,Bart Simpson
7,15.0,36623.0,C. Montgomery Burns
8,17.0,32855.0,Moe Szyslak
9,3.0,28138.0,Seymour Skinner
